# AgroLab AI — Pipeline de geração dos dados do morango
## Clima → NFT experimental + Solo convencional

**Cultura:** *Fragaria × ananassa* Duch. — cultivar `San Andreas`  
**Região de referência:** Bragança Paulista / Atibaia (SP)  
**Objetivo:** reproduzir, para o morango, a arquitetura usada no projeto do alface: uma base climática comum alimenta dois geradores sintéticos comparáveis.

### Saídas

- `clima/clima_processado.csv` e `clima/relatorio_clima.txt`;
- `nft/dataset_nft.csv` e `nft/relatorio_dataset_nft.txt`;
- `solo/dataset_solo.csv` e `solo/relatorio_dataset_solo.txt`.

As quatro classes seguem o projeto: `0 = não agir`, `1 = travar/corrigir excesso`, `2 = irrigar/repor falta`, `3 = proteger de extremos`.

## ⚠️ Delimitação metodológica

O arquivo meteorológico cobre apenas 32 dias entre dezembro de 2025 e janeiro de 2026. A pipeline agrega esse recorte horário e constrói um ano sintético sazonal, substituindo pelos valores observados os dias em que há correspondência. A coluna `origem_clima` diferencia os dois casos.

NFT e solo são **cenários sintéticos independentes**, ancorados na mesma série climática. Eles servem para desenvolver e testar o protótipo, não para concluir superioridade produtiva. O NFT de morango é tratado como cenário experimental; o sistema semi-hidropônico em substrato continua sendo um conjunto separado.

## Parte 1 — Configuração e caminhos

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
CENARIOS_POR_DIA = 15
rng = np.random.default_rng(SEED)

def localizar_morango():
    candidatos = [Path.cwd(), Path.cwd() / 'data/raw/morango']
    candidatos += [p / 'data/raw/morango' for p in Path.cwd().parents]
    for p in candidatos:
        if (p / 'clima' / 'dados-temp.csv').exists():
            return p.resolve()
    raise FileNotFoundError('Não encontrei data/raw/morango/clima/dados-temp.csv')

PASTA = localizar_morango()
DIR_CLIMA, DIR_NFT, DIR_SOLO = PASTA/'clima', PASTA/'nft', PASTA/'solo'
for p in (DIR_CLIMA, DIR_NFT, DIR_SOLO):
    p.mkdir(parents=True, exist_ok=True)

print('Pasta do morango:', PASTA)
print('Seed:', SEED, '| cenários por dia:', CENARIOS_POR_DIA)

Pasta do morango: C:\Users\Magora\Desktop\IC\ic-agro-lab\data\raw\morango
Seed: 42 | cenários por dia: 15


## Parte 2 — Pipeline climática

As observações horárias são convertidas em valores diários usando média para temperatura/umidade/vento, máximo e mínimo para extremos e soma para chuva/radiação. VPD e DLI são derivados por relações físicas declaradas.

In [5]:
ARQ_CLIMA_BRUTO = DIR_CLIMA / 'dados-temp.csv'
bruto = pd.read_csv(ARQ_CLIMA_BRUTO, sep=';', decimal=',', encoding='utf-8-sig', na_values=['', ' ', '-9999'])

# Renomeio por posição: evita diferenças de acentuação/mojibake entre sistemas.
nomes = ['data','hora_utc','temp_ins','temp_max','temp_min','ur_ins','ur_max','ur_min',
         'orvalho_ins','orvalho_max','orvalho_min','pressao_ins','pressao_max','pressao_min',
         'vento_ms','direcao_vento','rajada_ms','radiacao_kj_m2','chuva_mm']
if bruto.shape[1] != len(nomes):
    raise ValueError(f'Esperadas {len(nomes)} colunas meteorológicas; recebidas {bruto.shape[1]}')
bruto.columns = nomes
bruto['data'] = pd.to_datetime(bruto['data'], dayfirst=True, errors='raise')
for c in nomes[2:]:
    bruto[c] = pd.to_numeric(bruto[c], errors='coerce')

def pressao_saturacao(t):
    return 0.6108 * np.exp((17.27*t)/(t+237.3))

diario_obs = (bruto.groupby('data', as_index=False).agg(
    temp_ar_c=('temp_ins','mean'), temp_max_c=('temp_max','max'), temp_min_c=('temp_min','min'),
    umidade_relativa_pct=('ur_ins','mean'), ponto_orvalho_c=('orvalho_ins','mean'),
    chuva_mm=('chuva_mm','sum'), radiacao_kj_m2=('radiacao_kj_m2','sum'),
    vento_ms=('vento_ms','mean'), rajada_ms=('rajada_ms','max')))

diario_obs['vpd_kpa'] = (pressao_saturacao(diario_obs.temp_ar_c) -
                          pressao_saturacao(diario_obs.ponto_orvalho_c)).clip(lower=0)
diario_obs['dli_mol_m2_d'] = diario_obs.radiacao_kj_m2 * 0.45 * 4.57 / 1000
print('Horas:', len(bruto), '| dias observados:', len(diario_obs))
print(diario_obs.head(3).round(2).to_string(index=False))

Horas: 768 | dias observados: 32
<cell 5>:28: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
      data  temp_ar_c  temp_max_c  temp_min_c  umidade_relativa_pct  ponto_orvalho_c  chuva_mm  radiacao_kj_m2  vento_ms  rajada_ms  vpd_kpa  dli_mol_m2_d
2025-12-21      23.80        31.7        17.5                 71.08            17.13       0.0         25980.0      2.34       10.4     0.99         53.43
2025-12-22      25.09        32.0        20.1                 55.83            15.27       0.0         25332.5      2.33        8.8     1.45         52.10
2025-12-23      26.01        32.1        20.6                 54.25            15.61       0.0         24217.2      1.29        7.0     1.59         49.80


### 2.1 Expansão sazonal transparente para 365 dias

In [7]:
datas = pd.date_range('2026-01-01', '2026-12-31', freq='D')
doy = datas.dayofyear.to_numpy()
onda_verao = np.cos(2*np.pi*(doy-20)/365.25)  # máximo próximo de janeiro

clima = pd.DataFrame({'data': datas})
clima['cidade'] = 'Bragança Paulista/Atibaia'
clima['polo'] = 'Circuito das Frutas-SP'
clima['temp_ar_c'] = 19.0 + 3.8*onda_verao + rng.normal(0,1.4,len(datas))
amp = rng.uniform(4.5,7.5,len(datas))
clima['temp_max_c'] = clima.temp_ar_c + amp + rng.normal(0,.8,len(datas))
clima['temp_min_c'] = clima.temp_ar_c - amp + rng.normal(0,.8,len(datas))
clima['umidade_relativa_pct'] = np.clip(72 + 9*onda_verao + rng.normal(0,7,len(datas)),35,98)
clima['ponto_orvalho_c'] = clima.temp_ar_c - (100-clima.umidade_relativa_pct)/5
clima['chuva_mm'] = np.where(rng.random(len(datas)) < (0.18+0.18*(onda_verao+1)/2),
                             rng.gamma(1.6,5.0,len(datas)), 0)
clima['radiacao_kj_m2'] = np.clip(16500 + 5500*onda_verao + rng.normal(0,2800,len(datas)),3500,30000)
clima['vento_ms'] = np.clip(rng.normal(2.3,.8,len(datas)),0.1,None)
clima['rajada_ms'] = clima.vento_ms + rng.uniform(2,7,len(datas))
clima['origem_clima'] = 'sazonal_sintetico'

# Dias observados de 2026 substituem a sazonalidade, mantendo rastreabilidade.
obs_2026 = diario_obs[diario_obs.data.dt.year.eq(2026)].set_index('data')
cols_obs = ['temp_ar_c','temp_max_c','temp_min_c','umidade_relativa_pct','ponto_orvalho_c',
            'chuva_mm','radiacao_kj_m2','vento_ms','rajada_ms']
clima = clima.set_index('data')
idx = clima.index.intersection(obs_2026.index)
clima.loc[idx, cols_obs] = obs_2026.loc[idx, cols_obs]
clima.loc[idx, 'origem_clima'] = 'observado_horario_agregado'
clima = clima.reset_index()

clima['vpd_kpa'] = (pressao_saturacao(clima.temp_ar_c) - pressao_saturacao(clima.ponto_orvalho_c)).clip(lower=0)
clima['dli_mol_m2_d'] = clima.radiacao_kj_m2 * 0.45 * 4.57 / 1000
clima = clima.round({'temp_ar_c':2,'temp_max_c':2,'temp_min_c':2,'umidade_relativa_pct':2,
                     'ponto_orvalho_c':2,'chuva_mm':2,'radiacao_kj_m2':1,'vento_ms':2,
                     'rajada_ms':2,'vpd_kpa':3,'dli_mol_m2_d':2})

ARQ_CLIMA = DIR_CLIMA / 'clima_processado.csv'
clima.to_csv(ARQ_CLIMA, index=False, encoding='utf-8')
print('Clima salvo:', ARQ_CLIMA.relative_to(PASTA), '| linhas:', len(clima))
print(clima.origem_clima.value_counts().to_string())

Clima salvo: clima\clima_processado.csv | linhas: 365
origem_clima
sazonal_sintetico             344
observado_horario_agregado     21


In [8]:
rel_clima = [
    'AgroLab AI — RELATÓRIO DA PIPELINE CLIMÁTICA DO MORANGO',
    f'Período final: {clima.data.min().date()} a {clima.data.max().date()}',
    f'Linhas: {len(clima)} | dias observados incorporados: {(clima.origem_clima=="observado_horario_agregado").sum()}',
    'ATENÇÃO: dias restantes são sazonalidade sintética; não são observações de estação.',
    '', 'Resumo numérico:',
    clima[['temp_ar_c','temp_max_c','temp_min_c','umidade_relativa_pct','chuva_mm','vpd_kpa','dli_mol_m2_d']].describe().round(2).to_string()
]
(DIR_CLIMA/'relatorio_clima.txt').write_text('\n'.join(rel_clima), encoding='utf-8')
print('Relatório climático salvo.')

Relatório climático salvo.


## Parte 3 — Funções compartilhadas

Os dois sistemas recebem a mesma data, cultivar, fase fenológica e exposição climática. A comparação fica mais defensável porque a diferença vem das variáveis de manejo do sistema, não de climas distintos.

In [10]:
FASES = ['vegetativo','floracao','frutificacao','producao']

def adicionar_contexto(base, rng):
    n = len(base)
    dias = rng.integers(1,181,n)
    fase = np.select([dias<=35,dias<=65,dias<=110], FASES[:3], default=FASES[3])
    return dias, fase

def relatorio_dataset(ds, titulo, colunas):
    dist = ds.classe_acao.value_counts().sort_index()
    linhas = [titulo, '='*70, f'Linhas: {len(ds)} | colunas: {len(ds.columns)} | seed={SEED}',
              '', 'Distribuição das classes:']
    for c in range(4):
        q = int(dist.get(c,0)); linhas.append(f'  classe {c}: {q} ({100*q/len(ds):.1f}%)')
    linhas += ['', 'Coerência motivo × classe:', ds.groupby(['classe_acao','motivo']).size().to_string(),
               '', 'Estatísticas:', ds[colunas].describe().round(2).to_string()]
    return '\n'.join(linhas)

## Parte 4 — Gerador NFT experimental

No NFT, a CE representa a concentração iônica total; N, P, K, Ca e Mg escalam com ela. Temperatura da solução, oxigênio dissolvido, vazão e nível do reservatório são variáveis exclusivas do circuito hidropônico.

As faixas são hipóteses operacionais para simulação e devem ser validadas por especialista antes de controle real.

In [12]:
base_nft = clima.loc[clima.index.repeat(CENARIOS_POR_DIA)].reset_index(drop=True)
n = len(base_nft); dias, fase = adicionar_contexto(base_nft, rng)

alvos_ce = {'vegetativo':(0.9,1.2),'floracao':(1.1,1.4),'frutificacao':(1.3,1.6),'producao':(1.4,1.7)}
ce_min = np.array([alvos_ce[f][0] for f in fase]); ce_max = np.array([alvos_ce[f][1] for f in fase])
estado = rng.choice(['ideal','diluido','concentrado','ph_baixo','ph_alto','reservatorio_baixo'],
                   n,p=[.48,.15,.13,.07,.07,.10])
ce = rng.uniform(ce_min,ce_max)
ce[estado=='diluido'] = ce_min[estado=='diluido']*rng.uniform(.45,.9,(estado=='diluido').sum())
ce[estado=='concentrado'] = ce_max[estado=='concentrado']*rng.uniform(1.08,1.55,(estado=='concentrado').sum())
ph = rng.uniform(5.5,6.2,n)
ph[estado=='ph_baixo'] = rng.uniform(4.5,5.2,(estado=='ph_baixo').sum())
ph[estado=='ph_alto'] = rng.uniform(6.6,7.5,(estado=='ph_alto').sum())
nivel = rng.uniform(45,100,n); nivel[estado=='reservatorio_baixo'] = rng.uniform(5,25,(estado=='reservatorio_baixo').sum())

# Estufa: atenua frio/luz, mas pode elevar a máxima interna.
tmax = base_nft.temp_max_c.to_numpy()+1.5; tmin = base_nft.temp_min_c.to_numpy()+1.0
tmed = (tmax+tmin)/2; ur = np.clip(base_nft.umidade_relativa_pct.to_numpy()+3,0,100)
dli = base_nft.dli_mol_m2_d.to_numpy()*.72
tsol = np.clip(tmed + rng.normal(0,1.2,n),4,35)
od_sat = 14.62 - .3898*tsol + .006969*tsol**2 - .00005896*tsol**3
aeracao = rng.uniform(.72,1.0,n); od = np.clip(od_sat*aeracao,1,14)
vazao = np.clip(rng.normal(1.2,.25,n),.25,2.5)

fator = ce/1.5
refs = {'N':150,'P':40,'K':250,'Ca':120,'Mg':40}
nut = {k: np.clip(v*fator*rng.normal(1,.06,n),1,None) for k,v in refs.items()}

nft = pd.DataFrame({
    'timestamp':base_nft.data.dt.strftime('%Y-%m-%d'),'polo':base_nft.polo,
    'sistema':'hidroponia_nft_experimental','cultivar':'San Andreas',
    'dias_apos_transplante':dias,'fase':fase,'ce_ms_cm':ce.round(2),'ph':ph.round(2),
    **{k:v.round(1) for k,v in nut.items()},'temp_solucao_c':tsol.round(1),'od_mg_l':od.round(2),
    'vazao_l_min':vazao.round(2),'nivel_reservatorio_pct':nivel.round(1),
    'temp_ar_c':tmed.round(1),'temp_max_c':tmax.round(1),'temp_min_c':tmin.round(1),
    'umidade_relativa_pct':ur.round(1),'vpd_kpa':base_nft.vpd_kpa.round(2),
    'dli_mol_m2_d':dli.round(1),'origem_clima':base_nft.origem_clima,
    'ce_alvo_min':ce_min,'ce_alvo_max':ce_max})

frost=nft.temp_min_c<5; calor=nft.temp_max_c>32; raiz_quente=nft.temp_solucao_c>26; od_baixo=nft.od_mg_l<5
exc=(nft.ce_ms_cm>nft.ce_alvo_max)|(nft.ph>6.5)
falta=(nft.ce_ms_cm<nft.ce_alvo_min)|(nft.ph<5.3)|(nft.nivel_reservatorio_pct<25)|(nft.vazao_l_min<.5)
conds=[frost,calor,raiz_quente,od_baixo,exc,falta]
classes=[3,3,3,3,1,2]
motivos=['proteger: frio','proteger: calor do ar','proteger: solução quente','proteger: oxigênio dissolvido baixo',
         'travar/corrigir: CE ou pH alto','repor/corrigir: CE, pH, nível ou vazão baixos']
nft['classe_acao']=np.select(conds,classes,default=0).astype(int)
nft['motivo']=np.select(conds,motivos,default='condições dentro do ideal')
nft['saude_pct']=np.clip(100-18*(nft.classe_acao==3)-12*nft.classe_acao.isin([1,2]),0,100)
nft=nft.sample(frac=1,random_state=SEED).reset_index(drop=True)
print(nft.shape); print(nft.classe_acao.value_counts(normalize=True).sort_index().mul(100).round(1).to_string())

(5475, 29)
classe_acao
0    43.6
1    18.9
2    30.5
3     7.0


In [13]:
ARQ_NFT = DIR_NFT/'dataset_nft.csv'
nft.to_csv(ARQ_NFT,index=False,encoding='utf-8')
rel_nft=relatorio_dataset(nft,'AgroLab AI — MORANGO NFT EXPERIMENTAL',
    ['ce_ms_cm','ph','N','P','K','Ca','Mg','temp_solucao_c','od_mg_l','vazao_l_min','nivel_reservatorio_pct','saude_pct'])
(DIR_NFT/'relatorio_dataset_nft.txt').write_text(rel_nft,encoding='utf-8')
print('NFT salvo:',ARQ_NFT.relative_to(PASTA))

NFT salvo: nft\dataset_nft.csv


## Parte 5 — Gerador de solo convencional

No solo, N, P e K são proxies sintéticos de análise em `mg/dm³`; umidade representa água disponível. Temperatura do solo é derivada da média climática móvel. Os limiares são regras do protótipo, não recomendação de adubação.

In [15]:
base_solo=clima.loc[clima.index.repeat(CENARIOS_POR_DIA)].reset_index(drop=True)
n=len(base_solo); dias,fase=adicionar_contexto(base_solo,rng)
estado=rng.choice(['ideal','def_nutri','exc_nutri','seco','encharcado','ph_baixo','ph_alto'],
                  n,p=[.42,.13,.11,.13,.08,.065,.065])
N=rng.uniform(20,45,n); P=rng.uniform(30,80,n); K=rng.uniform(80,180,n)
ph=rng.uniform(5.5,6.5,n); umid=rng.uniform(55,90,n)

m=estado=='def_nutri'; escolha=rng.integers(0,3,m.sum()); idx=np.where(m)[0]
N[idx[escolha==0]]=rng.uniform(3,15,(escolha==0).sum()); P[idx[escolha==1]]=rng.uniform(5,20,(escolha==1).sum()); K[idx[escolha==2]]=rng.uniform(15,60,(escolha==2).sum())
m=estado=='exc_nutri'; escolha=rng.integers(0,3,m.sum()); idx=np.where(m)[0]
N[idx[escolha==0]]=rng.uniform(60,100,(escolha==0).sum()); P[idx[escolha==1]]=rng.uniform(120,220,(escolha==1).sum()); K[idx[escolha==2]]=rng.uniform(260,420,(escolha==2).sum())
umid[estado=='seco']=rng.uniform(15,45,(estado=='seco').sum()); umid[estado=='encharcado']=rng.uniform(101,125,(estado=='encharcado').sum())
ph[estado=='ph_baixo']=rng.uniform(4.3,5.0,(estado=='ph_baixo').sum()); ph[estado=='ph_alto']=rng.uniform(7.0,8.0,(estado=='ph_alto').sum())

temp_solo=np.clip(base_solo.temp_ar_c.to_numpy()+rng.normal(0,1,n),3,35)
solo=pd.DataFrame({
    'timestamp':base_solo.data.dt.strftime('%Y-%m-%d'),'polo':base_solo.polo,'sistema':'solo_convencional',
    'cultivar':'San Andreas','dias_apos_transplante':dias,'fase':fase,
    'N_mg_dm3':N.round(1),'P_mg_dm3':P.round(1),'K_mg_dm3':K.round(1),'ph':ph.round(2),
    'umidade_solo_pct':umid.round(1),'temp_solo_c':temp_solo.round(1),
    'temp_ar_c':base_solo.temp_ar_c.round(1),'temp_max_c':base_solo.temp_max_c.round(1),
    'temp_min_c':base_solo.temp_min_c.round(1),'umidade_relativa_pct':base_solo.umidade_relativa_pct.round(1),
    'vpd_kpa':base_solo.vpd_kpa.round(2),'dli_mol_m2_d':base_solo.dli_mol_m2_d.round(1),
    'chuva_mm':base_solo.chuva_mm.round(1),'origem_clima':base_solo.origem_clima})

frost=solo.temp_min_c<5; calor=solo.temp_max_c>32
exc=(solo.N_mg_dm3>50)|(solo.P_mg_dm3>100)|(solo.K_mg_dm3>235)|(solo.umidade_solo_pct>100)|(solo.ph>7)
falta=(solo.N_mg_dm3<15)|(solo.P_mg_dm3<20)|(solo.K_mg_dm3<60)|(solo.umidade_solo_pct<50)|(solo.ph<5.2)
conds=[frost,calor,exc,falta]; classes=[3,3,1,2]
motivos=['proteger: frio','proteger: calor do ar','travar/corrigir: excesso, encharcamento ou pH alto',
         'irrigar/adubar/corrigir: falta, seca ou pH baixo']
solo['classe_acao']=np.select(conds,classes,default=0).astype(int)
solo['motivo']=np.select(conds,motivos,default='condições dentro do ideal')
solo['saude_pct']=np.clip(100-18*(solo.classe_acao==3)-12*solo.classe_acao.isin([1,2]),0,100)
solo=solo.sample(frac=1,random_state=SEED).reset_index(drop=True)
print(solo.shape); print(solo.classe_acao.value_counts(normalize=True).sort_index().mul(100).round(1).to_string())

(5475, 23)
classe_acao
0    40.6
1    25.3
2    32.1
3     1.9


In [16]:
ARQ_SOLO=DIR_SOLO/'dataset_solo.csv'
solo.to_csv(ARQ_SOLO,index=False,encoding='utf-8')
rel_solo=relatorio_dataset(solo,'AgroLab AI — MORANGO EM SOLO CONVENCIONAL',
    ['N_mg_dm3','P_mg_dm3','K_mg_dm3','ph','umidade_solo_pct','temp_solo_c','temp_ar_c','saude_pct'])
(DIR_SOLO/'relatorio_dataset_solo.txt').write_text(rel_solo,encoding='utf-8')
print('Solo salvo:',ARQ_SOLO.relative_to(PASTA))

Solo salvo: solo\dataset_solo.csv


## Parte 6 — Validação cruzada das três saídas

In [18]:
checagens=pd.Series({
    'clima: 365 datas únicas':len(clima)==365 and clima.data.nunique()==365,
    'NFT: sem nulos':nft.isna().sum().sum()==0,
    'solo: sem nulos':solo.isna().sum().sum()==0,
    'NFT: quatro classes':set(nft.classe_acao)=={0,1,2,3},
    'solo: quatro classes':set(solo.classe_acao)=={0,1,2,3},
    'mesmo volume NFT e solo':len(nft)==len(solo)==365*CENARIOS_POR_DIA,
    'NFT: CE positiva':nft.ce_ms_cm.gt(0).all(),
    'solo: umidade fisicamente delimitada':solo.umidade_solo_pct.between(0,130).all(),
    'temperaturas ordenadas no clima':((clima.temp_min_c<=clima.temp_ar_c)&(clima.temp_ar_c<=clima.temp_max_c)).all(),
})
print(checagens.to_string())
assert checagens.all(),'Uma ou mais checagens falharam.'
print('\nPipeline concluída com sucesso.')

clima: 365 datas únicas                 True
NFT: sem nulos                          True
solo: sem nulos                         True
NFT: quatro classes                     True
solo: quatro classes                    True
mesmo volume NFT e solo                 True
NFT: CE positiva                        True
solo: umidade fisicamente delimitada    True
temperaturas ordenadas no clima         True

Pipeline concluída com sucesso.


## Parte 7 — Ordem de execução e próximos passos

1. Execute todas as células para reconstruir `clima_processado.csv`.
2. O mesmo clima alimentará NFT e solo, com 15 cenários por dia.
3. Os datasets e relatórios serão sobrescritos de forma reprodutível pela seed 42.
4. Antes do artigo, valide limiares com orientação agronômica e substitua a expansão sazonal por uma série oficial completa.
5. Adapte o EDA comparativo para ler `nft/dataset_nft.csv` e `solo/dataset_solo.csv`.

### Honestidade científica

- `origem_clima` permite separar dias observados de dias sintéticos;
- NPK do solo são proxies, não recomendação laboratorial;
- NFT é cenário experimental distinto da semi-hidroponia em substrato;
- `saude_pct` e `classe_acao` são alvos derivados por regra;
- nenhuma comparação causal de produtividade pode ser feita sem ensaio de campo.